# Making basic pipeline-based Approach

In [4]:
#All imports
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score ,root_mean_squared_error
import joblib
print("ALL imports Successfull")


ALL imports Successfull


In [ ]:
#All functions
path="C:\\Users\\sj81o\\Downloads\\archive (1)\\"
def load_data():
    players_data=pd.read_csv(path+"players.csv")
    del_data=pd.read_csv(path+"deliveries.csv")
    match_data=pd.read_csv(path+"matches.csv")
    
    return players_data,del_data,match_data
    
def clean_data(player_data,del_data,match_data):
    count_null_player=player_data.isna().sum().sum()
    count_null_del=del_data.isna().sum().sum()
    count_null_match=match_data.isna().sum().sum()
    
    print(f"player Data have Null values : {count_null_player}")
    print(f"Delievery Data have Null values : {count_null_del}")
    print(f"Match Data have Null values : {count_null_match}")
     

    return player_data,del_data,match_data
def data_featuring(players_data,del_data,match_data):
    runs= del_data.groupby("striker")["batsman_runs"].sum()
    ball=del_data.groupby("striker").size()
    dic={"Runs":runs ,"Ball_played":ball}
    score_frame=pd.DataFrame(dic)

    score_frame["strike_rate"]=(score_frame["Runs"]/score_frame["Ball_played"])*100
    score_frame=score_frame[score_frame["Ball_played"]>30]
    score_frame=score_frame.sort_values("strike_rate",ascending=False)
    score_frame["boundary_Percent"]=((del_data[del_data["batsman_runs"].isin([4,6])].groupby("striker").size())/score_frame['Ball_played'])*100
    score_frame["Dot_Percent"]=((del_data[del_data["batsman_runs"]==0].groupby("striker").size())/(score_frame["Ball_played"]))*100
    score_frame=score_frame.sort_values("strike_rate",ascending=False)
    merged=players_data.merge(score_frame,left_on="player_name",
                         right_on="striker",how="left")
    pom=match_data.groupby("player_of_match").size()
    pomdataframe=pd.DataFrame(pom)
    
    final=merged.merge(pomdataframe,left_on="player_name",right_on="player_of_match",how="left")
    final=final.rename(columns={0:"POM"})
    final["POM"] = final["POM"].fillna(0)
    return final

    

def train_model(final):
    features = [
    "strike_rate",
    "boundary_Percent",
    "Dot_Percent",
    "Runs",
    "POM",
    "is_capped_international",
    "playing_role",
    "base_price_lakh"
    ]
    x=final[features]
    y=final["highest_auction_price_lakh"]
    x=pd.get_dummies(x,columns=["playing_role"])
    ss=StandardScaler()
    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
    x_train=ss.fit_transform(x_train)
    x_test=ss.transform(x_test)
    joblib.dump(ss, "scaler.pkl")
    print("Standard scaler saved as scaler.pkl")
    model=LinearRegression()
    model.fit(x_train,y_train)
    y_pred=model.predict(x_test)
    joblib.dump(x.columns, "features.pkl")
    print("Features saves as  features.pkl")

    def evaluate_model():
        r2=r2_score(y_test,y_pred)
        rmse=root_mean_squared_error(y_test,y_pred)
        print(f"R2 Score : {r2}")
        print(f"RMSE Score : {rmse}")
    evaluate_model()
    return model
def save_model(model):
    joblib.dump(model,"model_pipe.pkl")
    print("Model saved....!")
    
def run_pipeline():
    player_data,del_data,match_data=load_data()
    player_data,del_data,match_data=clean_data( player_data,del_data,match_data)
    final=data_featuring(player_data,del_data,match_data)
    model=train_model(final)
    save_model(model)
    print("Pipeline executed successfully")

    


In [19]:
run_pipeline()

player Data have Null values : 287
Delievery Data have Null values : 507011
Match Data have Null values : 52
Standard scsaler saved as scaler.pkl
Features saves as  features.pkl
R2 Score : 0.560533681209477
RMSE Score : 356.48774006915664
Model saved....!
Pipeline executed successfully
